In [ ]:
# RF-DETR-Seg-Nano — Google Colab Training Pipeline

Dieses Notebook klont das Repository und führt die bestehenden Scripts der Reihe nach aus.

**Voraussetzung:** GPU-Laufzeit aktivieren: `Laufzeit → Laufzeittyp ändern → T4 GPU`

## 1. Repository klonen

In [ ]:
import os

REPO_URL = "https://github.com/Agredo/rf-detr-document.git"
REPO_DIR = "rf-detr-document"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL}")

os.chdir(REPO_DIR)
print(f"Arbeitsverzeichnis: {os.getcwd()}")
print("Repository-Inhalt:", os.listdir("."))

## 2. GPU prüfen

In [ ]:
import subprocess
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "⚠️ Keine GPU gefunden – bitte GPU-Laufzeit aktivieren!")

## 3. Abhängigkeiten installieren

In [ ]:
%pip install -q -r requirements.txt
print("✓ Installation abgeschlossen.")

## 4. Datensatz hochladen

Teile die ZIP lokal in ~500 MB Teile auf und lade sie nacheinander hoch:
```bash
# Lokal auf dem Mac ausführen:
zip -r train.zip "/Users/agredo/Doument Detection/train"
split -b 500m train.zip train_part_
```
Es entstehen Dateien wie `train_part_aa`, `train_part_ab`, ... — diese in der nächsten Zelle hochladen.

In [ ]:
import os, glob, zipfile, threading
from google.colab import files

PARTS_DIR = "zip_parts"
os.makedirs(PARTS_DIR, exist_ok=True)

# Alle train_part_* auf einmal auswählen — Browser lädt sie parallel hoch
print("Alle train_part_* Dateien auf einmal auswählen (Strg/Cmd+A oder Shift+Klick):")
uploaded = files.upload()

# Parallel auf Disk schreiben
def save_part(name, data):
    path = os.path.join(PARTS_DIR, name)
    with open(path, "wb") as f:
        f.write(data)
    print(f"  ✓ {name}  ({len(data)/1024/1024:.1f} MB)")

threads = [threading.Thread(target=save_part, args=(n, d)) for n, d in uploaded.items()]
for t in threads: t.start()
for t in threads: t.join()

# Teile zusammensetzen
parts = sorted(glob.glob(f"{PARTS_DIR}/train_part_*"))
print(f"\n{len(parts)} Teile gefunden: {[os.path.basename(p) for p in parts]}")

COMBINED_ZIP = "train.zip"
with open(COMBINED_ZIP, "wb") as out:
    for part in parts:
        with open(part, "rb") as f:
            out.write(f.read())
print(f"✓ train.zip zusammengesetzt ({os.path.getsize(COMBINED_ZIP)/1024/1024:.0f} MB)")

# Entpacken
EXTRACT_DIR = "roboflow_data/train"
os.makedirs(EXTRACT_DIR, exist_ok=True)
print(f"Entpacke nach {EXTRACT_DIR} ...")
with zipfile.ZipFile(COMBINED_ZIP, "r") as z:
    z.extractall(EXTRACT_DIR)

ann_files = glob.glob(f"{EXTRACT_DIR}/**/_annotations.coco.json", recursive=True)
ROBOFLOW_DIR = os.path.dirname(ann_files[0]) if ann_files else EXTRACT_DIR
print(f"✓ Datensatz bereit: {ROBOFLOW_DIR}  ({len(os.listdir(ROBOFLOW_DIR))} Dateien)")
%env ROBOFLOW_DIR={ROBOFLOW_DIR}

## 5. Datensatz vorbereiten

Führt `scripts/prepare_dataset.py` aus — normalisiert Kategorien und erstellt den 80/20 Train/Valid-Split.

In [ ]:
import subprocess, os, sys

roboflow_dir = os.environ.get("ROBOFLOW_DIR", "roboflow_data")

result = subprocess.run(
    [sys.executable, "scripts/prepare_dataset.py",
     "--roboflow-dir", roboflow_dir,
     "--out-dir", "data",
     "--val-split", "0.2",
     "--seed", "42"],
    text=True
)
if result.returncode != 0:
    raise RuntimeError("prepare_dataset.py fehlgeschlagen")

In [ ]:
## 6. Training

Führt `scripts/train.py` aus. Parameter hier anpassen:

import subprocess, sys

# ── Trainingsparameter ──────────────────
RUN_NAME = "v1"
EPOCHS   = "75"
BATCH    = "8"
LR       = "1e-4"
WORKERS  = "2"
IMG_SIZE = "312"
# ────────────────────────────────────────

result = subprocess.run(
    [sys.executable, "scripts/train.py",
     "--run-name",    RUN_NAME,
     "--epochs",      EPOCHS,
     "--batch",       BATCH,
     "--lr",          LR,
     "--workers",     WORKERS,
     "--img-size",    IMG_SIZE],
    text=True
)
if result.returncode != 0:
    raise RuntimeError("train.py fehlgeschlagen")

In [ ]:
## 7. ONNX Export

Führt `scripts/export_onnx.py` aus.

import subprocess, sys

RUN_NAME = "v1"  # muss mit Training übereinstimmen

result = subprocess.run(
    [sys.executable, "scripts/export_onnx.py",
     "--run-name", RUN_NAME,
     "--device",   "cpu"],
    text=True
)
if result.returncode != 0:
    raise RuntimeError("export_onnx.py fehlgeschlagen")

In [ ]:
## 8. Ergebnisse herunterladen

In [ ]:
from google.colab import files
import zipfile
from pathlib import Path

RUN_NAME = "v1"
run_dir  = Path("runs") / RUN_NAME
zip_path = f"{RUN_NAME}_results.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in run_dir.iterdir():
        z.write(f, f.name)

files.download(zip_path)
print(f"✓ {zip_path} wird heruntergeladen.")